In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ipywidgets import interact, widgets

df = pd.read_csv("job_salary_prediction_dataset.csv")

In [8]:
display(df.info())
display(df.describe())
display(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 10 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   job_title         250000 non-null  str  
 1   experience_years  250000 non-null  int64
 2   education_level   250000 non-null  str  
 3   skills_count      250000 non-null  int64
 4   industry          250000 non-null  str  
 5   company_size      250000 non-null  str  
 6   location          250000 non-null  str  
 7   remote_work       250000 non-null  str  
 8   certifications    250000 non-null  int64
 9   salary            250000 non-null  int64
dtypes: int64(4), str(6)
memory usage: 19.1 MB


None

,experience_years,skills_count,certifications,salary
count,250000.000000,250000.000000,250000.000000,250000.000000
mean,10.005408,9.997812,2.491928,145718.080524
std,6.060602,5.479288,1.706475,37407.952729
min,0.000000,1.000000,0.000000,31867.000000
25%,5.000000,5.000000,1.000000,119358.000000
50%,10.000000,10.000000,2.000000,143453.000000
75%,15.000000,15.000000,4.000000,169492.000000
max,20.000000,19.000000,5.000000,333046.000000


,job_title,experience_years,education_level,skills_count,industry,company_size,location,remote_work,certifications,salary
0,AI Engineer,10,Bachelor,2,Healthcare,Medium,India,Hybrid,2,109413
1,Data Analyst,5,Bachelor,17,Telecom,Small,Australia,No,0,93764
2,Frontend Developer,18,PhD,4,Media,Medium,Singapore,No,1,148123
3,Business Analyst,19,PhD,13,Retail,Medium,Canada,Yes,0,189123
4,Product Manager,15,Bachelor,7,Manufacturing,Large,Sweden,Yes,0,165069


In [9]:
plot_type = widgets.Dropdown(
    options=["Line", "Bar", "Scatter"],
    value="Line",
    description="Plot:"
)

x_col = widgets.Dropdown(
    options=df.columns,
    description="X:"
)

y_col = widgets.Dropdown(
    options=df.select_dtypes(include="number").columns,
    description="Y:"
)

group_check = widgets.Checkbox(
    value=True,
    description="Do Group"
)

agg_func = widgets.Dropdown(
    options=["mean", "sum", "min", "max", "count"],
    value="mean",
    description="Function:"
)

sort_check = widgets.Checkbox(
    value=False,
    description="Sort"
)

save_button = widgets.Button(
    description="Save graph",
    button_style="success"
)

sort_order = widgets.ToggleButtons(
    options=["Ascending", "Descending"],
    description="Order:"
)

range_slider = widgets.IntRangeSlider(
    value=[0, min(100, len(df))],
    min=0,
    max=len(df),
    step=1,
    description="Range:",
    continuous_update=False
)

ui = widgets.HBox([
    plot_type,
    x_col,
    y_col,
    group_check,
    agg_func,
    sort_check,
    save_button,
    sort_order,
    range_slider
])

In [10]:
def update(plot_type,
           x_col,
           y_col,
           group_check,
           agg_func,
           sort_check,
           sort_order,
           range_slider):

    plt.figure(figsize=(10, 6))

    start, end = range_slider
    filtered_df = df.iloc[start:end]

    match plot_type:
        case "Line":
            func = plt.plot
        case "Bar":
            func = plt.bar
        case "Scatter":
            func = plt.scatter
        case _:
            return

    if group_check:

        group = getattr(
            filtered_df.groupby(x_col)[y_col],
            agg_func
        )()

        if sort_check:
            ascending = sort_order == "Ascending"
            group = group.sort_values(
                ascending=ascending
            )

        func(group.index, group.values)

        records = len(group)

    else:

        show_df = filtered_df

        if sort_check:
            ascending = sort_order == "Ascending"
            show_df = show_df.sort_values(
                y_col,
                ascending=ascending
            )

        func(
            show_df[x_col],
            show_df[y_col]
        )

        records = len(show_df)
        
    plt.title(
        f"{plot_type} plot of {y_col} by {x_col}\n"
        f"Records: {records}"
    )

    plt.xlabel(x_col)
    plt.ylabel(y_col)

    plt.xticks(rotation=90)

    plt.grid(True)

    plt.tight_layout()

    plt.show()


def save_graph(button):

    plt.figure(figsize=(10, 6))

    start, end = range_slider.value
    filtered_df = df.iloc[start:end]

    if group_check.value:

        group = getattr(
            filtered_df.groupby(x_col.value)[y_col.value],
            agg_func.value
        )()

        if sort_check.value:
            ascending = (
                sort_order.value == "Ascending"
            )
            group = group.sort_values(
                ascending=ascending
            )

        plt.bar(
            group.index,
            group.values
        )

    else:

        show_df = filtered_df

        if sort_check.value:
            ascending = (
                sort_order.value == "Ascending"
            )
            show_df = show_df.sort_values(
                y_col.value,
                ascending=ascending
            )

        plt.bar(
            show_df[x_col.value],
            show_df[y_col.value]
        )

    plt.title("Saved Graph")

    plt.xticks(rotation=90)

    plt.tight_layout()

    plt.savefig("graph.png")

save_button.on_click(save_graph)

In [11]:
out = widgets.interactive_output(
    update,
    {
        "plot_type": plot_type,
        "x_col": x_col,
        "y_col": y_col,
        "group_check": group_check,
        "agg_func": agg_func,
        "sort_check": sort_check,
        "sort_order": sort_order,
        "range_slider": range_slider
    }
)

In [12]:
display(ui, out)

Output(outputs=({'output_type': 'display_data', 'data': {'text/plain': '<Figure size 1000x600 with 1 Axes>', '…